# SPCE5025 Week 15 Homework

**Given** the following orbital state (at epoch 2012-10-04 05:00:44.523):

| Parameter       | Value                  | Units     |
|-----------------|------------------------|-----------|
| **X**           | 26,324,639.619         | m         |
| **Y**           | -5,219,093.894         | m         |
| **Z**           | -0.875                 | m         |
| **XD**          | 430.272110             | m/s       |
| **YD**          | 2,169.355094           | m/s       |
| **ZD**          | 3,156.244863           | m/s       |
| **A** (SMA)     | 26,837,890.921         | m         |
| **E**           | 0.000056               | -         |
| **I**           | 54.980669              | deg       |
| **RAAN**        | 348.786022             | deg       |
| **WP** (ω)      | 305.630097             | deg       |
| **NU** (ν)      | 54.369901              | deg       |
| **MA**          | 54.364707              | deg       |
| **TP**          | 729.1724               | min       |
| **GLAN**        | 80.4704                | deg       |
| **ALTD**        | 20,458.88              | km        |

---

## Find / Compute:

1. **ΔV required to change inclination by 5 degrees**  
   (pure plane change)

2. Apply the impulsive **ΔV** (in the correct direction) to the velocity vector and compute the **post-burn Keplerian elements** to verify the result.

3. **ΔV required to change SMA by 26 km**  
   (in-plane tangential burn)

4. Apply the impulsive **ΔV** in the velocity direction and compute the new Keplerian elements to verify.

5. **ΔSMA resulting from a ΔV = 1 m/s** (in the velocity direction).

6. **Phase rate** between the given orbit and a second orbit with period = **728.0 minutes**.

7. **Time required** for the phase angle to change by **30 degrees**.

---

**Notes:**
- The orbit is nearly circular (`e ≈ 0.000056`).
- Use the given position and velocity vectors for impulsive burn applications.
- For plane change, remember the optimal direction is along the nodal line (but for pure inclination change at this point, compute accordingly).
- You may use both analytical approximations from class **and** numerical verification by applying ΔV to the state vector.

In [58]:
from standards import *
epoch = datetime(2012, 10, 4, 5, 0, 44, 523000)   # 523 microseconds
pos = Vector3(x=26324639.619,y=-5219093.894,z=-0.875)
vel = Vector3(430.272110,2169.355094,3156.244863)
a = 26837890.921
e = 0.000056
i = 54.980669
raan = 348.786022
argp = 305.630097
nu = 54.369901
ma = 54.364707
tp = 729.1724
glan = 80.4704
altd = 20458.88
period_sec = tp * 60.0
mu = 3.986004418e14

#**ΔV required to change inclination by 5 degrees**  
v_orbit = vel.magnitude()
delta_v_5_deg_plane_change = 2*v_orbit*sin(radians(5)/2)
print(f"Delta v for 5 deg plane change: {delta_v_5_deg_plane_change:.6f} m/sec")

Delta v for 5 deg plane change: 336.215988 m/sec


In [59]:
#Apply the impulsive **ΔV** (in the correct direction) to the velocity vector and compute the **post-burn Keplerian elements** to verify the result.
final_inc_rad = radians(i+5)
ang_momentum_post_man_unit = Vector3(sin(final_inc_rad)*sin(radians(raan)), -sin(final_inc_rad)*cos(radians(raan)), cos(final_inc_rad))
vel_post_man_unit = ang_momentum_post_man_unit.cross(pos)/(ang_momentum_post_man_unit.magnitude()*pos.magnitude())
vel_post_man = vel.magnitude()*vel_post_man_unit
print(f"Velocity post maneuver: {vel_post_man}")
orig_kep = KeplerianElements(pos, vel)
post_man_kep = KeplerianElements(pos, vel_post_man)
# Calculate differences
diff_a = post_man_kep.a - orig_kep.a
diff_e = post_man_kep.ecc - orig_kep.ecc
diff_inc = post_man_kep.inc_deg - orig_kep.inc_deg
diff_raan = post_man_kep.raan_deg - orig_kep.raan_deg
diff_wp = post_man_kep.argp_deg - orig_kep.argp_deg
diff_nu = post_man_kep.ta_deg - orig_kep.ta_deg

print("5 deg inc change maneuver:")
print(" " * 12 + "Pre-burn" + " " * 18 + "Diff" + " " * 18 + "Post-burn")
print(f"a:      {orig_kep.a:15.6f}          {diff_a:12.6f}          {post_man_kep.a:15.6f} m")
print(f"e:      {orig_kep.ecc:15.8f}          {diff_e:12.8f}           {post_man_kep.ecc:15.8f}")
print(f"inc:    {orig_kep.inc_deg:15.6f}          {diff_inc:12.6f}          {post_man_kep.inc_deg:15.6f} deg")
print(f"O:      {orig_kep.raan_deg:15.6f}          {diff_raan:12.6f}          {post_man_kep.raan_deg:15.6f} deg")
print(f"wp:     {orig_kep.argp_deg:15.6f}          {diff_wp:12.6f}          {post_man_kep.argp_deg:15.6f} deg")
print(f"nu:     {orig_kep.ta_deg:15.6f}          {diff_nu:12.6f}          {post_man_kep.ta_deg:15.6f} deg")

Velocity post maneuver: Vector3(x=374.9673559795079, y=1891.3010180813283, z=3336.989240119866)
5 deg inc change maneuver:
            Pre-burn                  Diff                  Post-burn
a:      26837890.916314              0.000000          26837890.916314 m
e:           0.00005576           -0.00002328                0.00003249
inc:          54.980669              5.000000                59.980669 deg
O:           -11.213978             -0.000000               -11.213978 deg
wp:          -54.369965             54.369963                -0.000002 deg
nu:           54.369963            -54.369963                 0.000000 deg


In [60]:
#**ΔV required to change SMA by 26 km**  
delta_v_est = pi/(tp*60)*(26*1000)
print(f"estimated delta_v required to change SMA by 26 km: {delta_v_est:.4f} m/s")

v_pre  = sqrt((mu/a)* ((1 + 2*e*cos(radians(nu)) + e**2) / (1 - e**2)))
v_post = sqrt((mu/(a + 26000)) * ((1 + 2*e*cos(radians(nu)) + e**2) / (1 - e**2)))
delta_v = v_pre-v_post
print(f"actual delta_v (with eccentricity) required to change SMA by 26 km: {delta_v:.4f} m/s")

estimated delta_v required to change SMA by 26 km: 1.8670 m/s
actual delta_v (with eccentricity) required to change SMA by 26 km: 1.8655 m/s


In [61]:
# Apply the impulsive **ΔV** in the velocity direction and compute the new Keplerian elements to verify.
vel_unit = vel/vel.magnitude()
vel_post = vel + delta_v_est*vel_unit
orig_kep = KeplerianElements(pos, vel)
post_man_kep = KeplerianElements(pos, vel_post)
# Calculate differences
diff_a = post_man_kep.a - orig_kep.a
diff_e = post_man_kep.ecc - orig_kep.ecc
diff_inc = post_man_kep.inc_deg - orig_kep.inc_deg
diff_raan = post_man_kep.raan_deg - orig_kep.raan_deg
diff_wp = post_man_kep.argp_deg - orig_kep.argp_deg
diff_nu = post_man_kep.ta_deg - orig_kep.ta_deg

print("26 km sma diff maneuver:")
print(" " * 12 + "Pre-burn" + " " * 18 + "Diff" + " " * 18 + "Post-burn")
print(f"a:      {orig_kep.a:15.6f}          {diff_a:12.6f}          {post_man_kep.a:15.6f} m")
print(f"e:      {orig_kep.ecc:15.8f}          {diff_e:12.8f}           {post_man_kep.ecc:15.8f}")
print(f"inc:    {orig_kep.inc_deg:15.6f}          {diff_inc:12.6f}          {post_man_kep.inc_deg:15.6f} deg")
print(f"O:      {orig_kep.raan_deg:15.6f}          {diff_raan:12.6f}          {post_man_kep.raan_deg:15.6f} deg")
print(f"wp:     {orig_kep.argp_deg:15.6f}          {diff_wp:12.6f}          {post_man_kep.argp_deg:15.6f} deg")
print(f"nu:     {orig_kep.ta_deg:15.6f}          {diff_nu:12.6f}          {post_man_kep.ta_deg:15.6f} deg")

26 km sma diff maneuver:
            Pre-burn                  Diff                  Post-burn
a:      26837890.916314          26035.494210          26863926.410525 m
e:           0.00005576            0.00094688                0.00100264
inc:          54.980669             -0.000000                54.980669 deg
O:           -11.213978             -0.000000               -11.213978 deg
wp:          -54.369965             51.776603                -2.593362 deg
nu:           54.369963            -51.776603                 2.593360 deg


In [62]:
#**ΔSMA resulting from a ΔV = 1 m/s** (in the velocity direction).
delta_v = 1 #m/s intrack direction
delta_sma = delta_v/(pi/(tp*60))
print(f"delta sma after 1 m/s delta v: {delta_sma:.3f} m")

delta sma after 1 m/s delta v: 13926.167 m


In [63]:
#**Phase rate** between the given orbit and a second orbit with period = **728.0 minutes**.
phase_rate_change = 2*pi*(1/(728*60) - 1/(tp*60))
#convert rad/sec to deg/day
phase_rate_change = degrees(phase_rate_change)*60*60*24
print(f"Phase Angle Rate: {phase_rate_change:.4f} deg/day")

Phase Angle Rate: 1.1449 deg/day


In [64]:
#**Time required** for the phase angle to change by **30 degrees**.
time_required = 1/(phase_rate_change*(1/30))
print(f"time required: {time_required:.6f} days")

time required: 26.202459 days
